# IIG0029 — Técnicas de IA I
## Aula 04 — Laboratório: Preparação de Dados, Nulos e Escalonamento

**Professor:** Paulo Victor dos Santos  
**Dataset:** Titanic (pandas / Kaggle)  

Este notebook cobre as três atividades da aula:

1. Investigação de nulos e classificação MCAR / MAR / MNAR  
2. Comparação de imputações em `Age` (remoção, mediana, KNN)  
3. `StandardScaler` versus `MinMaxScaler` em colunas com outliers (`Fare`)

> Regra de ouro da aula: **primeiro** `train_test_split`, **depois** `fit()` só no treino.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

URL = "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

---
## Atividade 1 — Investigação de nulos

**Pedido:** calcular a porcentagem exata de valores ausentes por coluna e classificar a ausência em `Age` e `Cabin` como MCAR, MAR ou MNAR, com justificativa no contexto do desastre.

In [ ]:
nulos = df.isna().sum()
pct = 100 * nulos / len(df)

relatorio_nulos = pd.DataFrame({
    "nulos": nulos,
    "total": len(df),
    "pct": pct.round(4),
}).sort_values("pct", ascending=False)

relatorio_nulos

In [ ]:
print("=== Age ausente (%) por Pclass ===")
print((100 * df.groupby("Pclass")["Age"].apply(lambda s: s.isna().mean())).round(2))

print("\n=== Age ausente (%) por Sex ===")
print((100 * df.groupby("Sex")["Age"].apply(lambda s: s.isna().mean())).round(2))

print("\n=== Age ausente (%) por Survived ===")
print((100 * df.groupby("Survived")["Age"].apply(lambda s: s.isna().mean())).round(2))

print("\n=== Cabin ausente (%) por Pclass ===")
print((100 * df.groupby("Pclass")["Cabin"].apply(lambda s: s.isna().mean())).round(2))

print("\n=== Cabin ausente (%) por Survived ===")
print((100 * df.groupby("Survived")["Cabin"].apply(lambda s: s.isna().mean())).round(2))

### Resposta da Atividade 1 (para entrega)

| Coluna | Nulos | % |
|---|---|---|
| Cabin | 687 | **77,1044%** |
| Age | 177 | **19,8653%** |
| Embarked | 2 | **0,2245%** |
| demais | 0 | 0% |

**Age → MAR (Missing at Random)**  
A falta de idade **não parece aleatória**: concentra-se na 3ª classe (~27,7%) e é bem menor na 2ª (~6,0%). Também é um pouco maior entre homens e entre quem **não** sobreviveu. No contexto do Titanic, passageiros de 3ª classe tinham registro menos completo (embarque em massa, menos documentação). A ausência depende de variáveis **já observadas** (`Pclass`, `Sex`, `Survived`), não necessariamente do valor da idade em si — típico de MAR.

**Cabin → MAR (com forte viés de classe; não é MCAR)**  
Quase todos da 2ª e 3ª classe não têm cabine registrada (91% e 98%), enquanto na 1ª classe só ~19% falta. Cabines eram sobretudo de 1ª classe; a falta está ligada a `Pclass` (observada).  
Não classificamos como MNAR “puro” porque o valor omitido não é “quem tem cabine X esconde o número”, e sim “quem não tinha cabine / não foi registrado”. Na prática de ML, `Cabin` é pouco confiável (~77% nulo) e a aula recomenda **descartar** a coluna.

**Por que não MCAR?** Se fosse MCAR, a taxa de nulos seria parecida em todas as classes. Não é.

---
## Atividade 2 — Análise de imputações em `Age`

Três estratégias:

- **(a)** remover linhas com `Age` nulo  
- **(b)** imputar com a **mediana** (`SimpleImputer`)  
- **(c)** imputar com **KNN** (`KNNImputer`, k=5)

Compare graficamente a distribuição das idades **antes** e **depois**.

In [ ]:
age_original = df["Age"].dropna()
print(f"Linhas totais: {len(df)}")
print(f"Idades observadas (não nulas): {len(age_original)}")
print(f"Nulos em Age: {df['Age'].isna().sum()}")
print(age_original.describe())

In [ ]:
# (a) Eliminação de linhas
df_drop = df.dropna(subset=["Age"])
age_drop = df_drop["Age"]

# (b) Mediana — fit só no treino para não vazar estatística do teste
X_num = df[["Age", "Pclass", "Fare", "SibSp"]].copy()
X_train, X_test = train_test_split(X_num, test_size=0.2, random_state=42)

imputer_med = SimpleImputer(strategy="median")
X_train_med = imputer_med.fit_transform(X_train)
X_test_med = imputer_med.transform(X_test)

age_mediana_full = SimpleImputer(strategy="median").fit_transform(X_num)[:, 0]
print("Mediana usada (treino):", imputer_med.statistics_[0])
print("Mediana usada (dataset completo, só p/ o gráfico da atividade):",
      np.nanmedian(X_num["Age"]))

# (c) KNN
knn = KNNImputer(n_neighbors=5)
X_knn = knn.fit_transform(X_num)
age_knn = X_knn[:, 0]

print("\nMédia Age original :", round(age_original.mean(), 2))
print("Média Age mediana  :", round(np.mean(age_mediana_full), 2))
print("Média Age KNN      :", round(np.mean(age_knn), 2))
print("Desvio original    :", round(age_original.std(), 2))
print("Desvio mediana     :", round(np.std(age_mediana_full, ddof=1), 2))
print("Desvio KNN         :", round(np.std(age_knn, ddof=1), 2))
print(f"\n(a) linhas restantes após dropna: {len(df_drop)} de {len(df)}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

series = [
    (age_original, "Original (só observados)"),
    (age_drop, "(a) Remoção de linhas"),
    (age_mediana_full, "(b) Mediana"),
    (age_knn, "(c) KNN (k=5)"),
]

for ax, (dados, titulo) in zip(axes, series):
    ax.hist(dados, bins=30, edgecolor="white")
    ax.axvline(np.median(dados), color="black", linestyle="--", label="mediana")
    ax.set_title(titulo)
    ax.set_xlabel("Idade")
    ax.legend()

axes[0].set_ylabel("Frequência")
fig.suptitle("Distribuição de Age: original vs. três tratamentos de nulos", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(
    [age_original, age_drop, age_mediana_full, age_knn],
    tick_labels=["Original", "(a) Drop", "(b) Mediana", "(c) KNN"],
)
ax.set_ylabel("Idade")
ax.set_title("Boxplot de Age por método de tratamento")
plt.tight_layout()
plt.show()

### Resposta da Atividade 2 (para entrega)

- **(a) Remoção:** a forma do histograma de `Age` **não muda** (são os mesmos 714 valores observados), mas o dataset cai de 891 para 714 linhas (~20% de perda). Se a falta for MAR, isso **enviesa** o treino (some mais gente da 3ª classe).
- **(b) Mediana:** preserva 891 linhas, mas cria um **pico artificial** na mediana (~28 anos): todos os 177 nulos viram o mesmo número. A variância da idade **cai**.
- **(c) KNN:** também preserva 891 linhas. Cada nulo recebe a média dos 5 vizinhos mais parecidos em `Pclass`, `Fare` e `SibSp`. A distribuição fica **mais próxima** da original, sem o pico da mediana.

Para o relatório: o gráfico (b) distorce a forma; o (a) distorce o tamanho e o viés da amostra; o (c) é o melhor compromisso neste laboratório.

---
## Atividade 3 — StandardScaler vs MinMaxScaler (outliers)

`Fare` é contínua e **muito** sensível a outliers (máximo 512 vs. mediana ~14).  
Vamos: (1) ver como cada scaler comprime os valores; (2) uma regressão simples `Survived ~ Age + Fare` para ver o impacto nos coeficientes.

In [ ]:
cols = ["Age", "Fare"]
Xy = df[cols + ["Survived"]].copy()
Xy[cols] = SimpleImputer(strategy="median").fit_transform(Xy[cols])

X = Xy[cols]
y = Xy["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X["Fare"].describe())
print("\nRazão max/mediana de Fare:", round(X["Fare"].max() / X["Fare"].median(), 1))

In [ ]:
std = StandardScaler()
mm = MinMaxScaler()

X_train_std = std.fit_transform(X_train)
X_test_std = std.transform(X_test)

X_train_mm = mm.fit_transform(X_train)
X_test_mm = mm.transform(X_test)

fare_raw = X_train["Fare"].to_numpy()
fare_std = X_train_std[:, 1]
fare_mm = X_train_mm[:, 1]

print("MinMax  — min/max de Fare no treino:", fare_mm.min(), fare_mm.max())
print("Standard — média/desvio de Fare no treino:", round(fare_std.mean(), 4), round(fare_std.std(), 4))
print("% de tarifas MinMax < 0.10 (quase coladas no zero):",
      round(100 * (fare_mm < 0.10).mean(), 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(fare_raw, bins=40, edgecolor="white")
axes[0].set_title("Fare bruta (treino)")
axes[0].set_xlabel("Tarifa")

axes[1].hist(fare_mm, bins=40, edgecolor="white")
axes[1].set_title("Fare após MinMaxScaler [0, 1]")
axes[1].set_xlabel("Fare normalizada")

axes[2].hist(fare_std, bins=40, edgecolor="white")
axes[2].set_title("Fare após StandardScaler")
axes[2].set_xlabel("Fare padronizada (z-score)")

fig.suptitle("O outlier de Fare (~512) espreme o Min-Max perto de zero", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_train["Age"], X_train["Fare"], alpha=0.45)
axes[0].set_title("Nuvem bruta: Age vs Fare")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Fare")

axes[1].scatter(X_train_mm[:, 0], X_train_mm[:, 1], alpha=0.45, label="MinMax")
axes[1].scatter(X_train_std[:, 0], X_train_std[:, 1], alpha=0.45, label="Standard")
axes[1].set_title("Mesmos pontos após cada scaler")
axes[1].set_xlabel("Age escalonada")
axes[1].set_ylabel("Fare escalonada")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Correlação Age–Fare (bruta)   :", round(X_train.corr().loc["Age", "Fare"], 4))
print("Correlação Age–Fare (MinMax)  :", round(np.corrcoef(X_train_mm[:, 0], X_train_mm[:, 1])[0, 1], 4))
print("Correlação Age–Fare (Standard):", round(np.corrcoef(X_train_std[:, 0], X_train_std[:, 1])[0, 1], 4))
print("\nA correlação linear é invariante a escala afim — os dois scalers dão o MESMO r.")
print("Por isso o experimento útil é a REGRESSÃO e o histograma, não o heatmap de correlação.")

In [ ]:
def treinar(Xtr, Xte, nome):
    modelo = LinearRegression()
    modelo.fit(Xtr, y_train)
    pred = modelo.predict(Xte)
    print(f"\n{nome}")
    print("  coef Age :", round(modelo.coef_[0], 5))
    print("  coef Fare:", round(modelo.coef_[1], 5))
    print("  |Fare|/|Age|:", round(abs(modelo.coef_[1]) / abs(modelo.coef_[0]), 2))
    print("  R² teste :", round(r2_score(y_test, pred), 4))
    return modelo

print("Sem escala (valores brutos — Fare 'ganha' porque a unidade é maior):")
treinar(X_train, X_test, "Bruto")
treinar(X_train_mm, X_test_mm, "MinMaxScaler")
treinar(X_train_std, X_test_std, "StandardScaler")

### Resposta da Atividade 3 (para entrega)

- **MinMaxScaler** mapeia o mínimo para 0 e o máximo para 1. Um único `Fare=512` vira 1,0 e **espreme** a maioria das tarifas (mediana ~14) para perto de zero. A **variabilidade real** da massa de passageiros fica comprimida — histograma com barra enorme em ~0.
- **StandardScaler** centra em média 0 e desvio 1. Outliers viram z-scores altos, mas **não forçam** um teto. A forma da distribuição se mantém; a massa de dados continua espalhada em unidades de desvio padrão.
- A **correlação de Pearson** entre Age e Fare **não muda** (transformação linear). Por isso o gráfico de correlação sozinho **não** mostra quem distorce menos — use histograma + regressão.
- Na regressão, sem escala os coeficientes não são comparáveis (unidades diferentes). Com StandardScaler, cada coeficiente é “efeito de 1 desvio padrão”; com MinMax, “efeito de ir do min ao max”. O MinMax faz o coeficiente de `Fare` parecer enorme porque o eixo inteiro é dominado pelo outlier.

**Conclusão da aula:** para `Fare` (e a maioria das tarefas clássicas de ML), **StandardScaler distorce menos** a variabilidade da maior parte dos dados. Min-Max só é preferível quando o algoritmo exige intervalo fechado (ex.: pixels [0, 1]) **e** os outliers já foram tratados.

---
## Extra — pipeline da aula (split → one-hot → imputar → escalonar)

Mesmo script dos slides, para conferir no Colab.

In [ ]:
X = df[["Age", "Pclass", "Fare", "Sex"]].copy()
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = pd.get_dummies(X_train, columns=["Sex"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["Sex"], drop_first=True)

imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_imputed)
X_test_final = scaler.transform(X_test_imputed)

print("Treino preparado (shape):", X_train_final.shape)
print("Teste preparado (shape):", X_test_final.shape)
print("Nulos no treino após imputação:", np.isnan(X_train_final).sum())